# Student Dropout Prediction - Stage 3


## Colab / Local Setup

Local execution still works from the cloned repo in VS Code / WSL.

In Google Colab, start each new runtime by cloning the repo into `/content/student-dropout-prediction`, installing dependencies, and then running the setup cell below:

```bash
!git clone <repo-url> /content/student-dropout-prediction
%cd /content/student-dropout-prediction
!pip install -r requirements.txt
```

If you clone into a different folder name, set `COLAB_PROJECT_REPO` before running the setup cell.


In [ ]:
import os
import sys
from pathlib import Path


def _is_colab_runtime() -> bool:
    return 'google.colab' in sys.modules


def _resolve_repo_root() -> Path:
    repo_name = os.environ.get('COLAB_PROJECT_REPO', 'student-dropout-prediction')
    search_roots = [Path.cwd().resolve()]

    if _is_colab_runtime():
        colab_repo_root = Path('/content') / repo_name
        search_roots.append(colab_repo_root)

    seen = set()
    for root in search_roots:
        for candidate in [root, *root.parents]:
            if candidate in seen:
                continue
            seen.add(candidate)
            if all((candidate / part).exists() for part in ('src', 'data', 'notebooks')):
                return candidate

    raise FileNotFoundError(
        'Could not locate the repository root. In Colab, clone the repo into '
        f'/content/{repo_name} or set COLAB_PROJECT_REPO to the cloned folder name.'
    )


IS_COLAB = _is_colab_runtime()
BOOTSTRAP_PROJECT_ROOT = _resolve_repo_root()
BOOTSTRAP_NOTEBOOK_DIR = BOOTSTRAP_PROJECT_ROOT / 'notebooks'

if IS_COLAB and Path.cwd().resolve() != BOOTSTRAP_NOTEBOOK_DIR:
    os.chdir(BOOTSTRAP_NOTEBOOK_DIR)

if str(BOOTSTRAP_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_PROJECT_ROOT))

print(f'Running in Colab: {IS_COLAB}')
print(f'Project root: {BOOTSTRAP_PROJECT_ROOT}')
print(f'Working directory: {Path.cwd().resolve()}')


In [ ]:
import json
import os
import random
import ssl
import sys
from pathlib import Path

import joblib
import keras_tuner as kt
import matplotlib.pyplot as plt
import numpy as np
import optuna
import optuna.visualization as vis
import pandas as pd
import requests
import seaborn as sns
import shap
import tensorflow as tf
import xgboost as xgb
from IPython.display import HTML, display
from sklearn.metrics import mean_squared_error, r2_score
from tensorflow.keras import mixed_precision
from tensorflow.keras.callbacks import EarlyStopping as KerasEarlyStopping
from xgboost.callback import EarlyStopping as XgbEarlyStopping

from src.config import (
    DEFAULT_RANDOM_SEED,
    MANUAL_REVIEW_DEFAULT,
    RUNTIME_DEFAULTS,
    STABILITY_SEEDS,
    TARGET_COLUMN,
    TUNING_ARTIFACT_FILENAMES,
)
from src.notebook_helpers import (
    analyze_hp_importance,
    build_binary_classifier,
    build_models_to_plot,
    choose_n_jobs,
    compute_confusion_matrix_elements,
    compute_performance_metrics,
    create_train_val_test_split_and_scale,
    evaluate_and_store_model,
    plot_confusion_matrix,
    plot_model_confusion_matrix,
    plot_grouped_feature_importance,
    plot_roc_and_pr_curves,
    print_model_metrics,
    set_seed,
)
from src.paths import (
    DATA_DIR,
    FIGURES_DIR,
    MODELS_DIR,
    PROJECT_ROOT,
    TUNING_DIR,
    ensure_directories,
    get_stage_data_path,
    get_stage_paths,
)
from src.tuning_utils import (
    best_params_record,
    load_model_weights,
    load_saved_best_params,
    run_keras_tuner as shared_run_keras_tuner,
    run_optuna_xgb as shared_run_optuna_xgb,
    tuner_trials_to_dataframe,
)

STAGE_NAME = 'stage_3'
PREVIOUS_STAGE = 'stage_2'
stage_paths = get_stage_paths(STAGE_NAME, previous_stage=PREVIOUS_STAGE)
STAGE_DATA_PATH = get_stage_data_path(STAGE_NAME)
DATA_CACHE_DIR = stage_paths['DATA_CACHE_DIR']
STAGE_TUNING_DIR = stage_paths['STAGE_TUNING_DIR']
XGB_TUNING_DIR = stage_paths['XGB_TUNING_DIR']
NN_TUNING_DIR = stage_paths['NN_TUNING_DIR']
PREV_XGB_TUNING_DIR = stage_paths['PREV_XGB_TUNING_DIR']
PREV_NN_TUNING_DIR = stage_paths['PREV_NN_TUNING_DIR']
STAGE_MODEL_DIR = stage_paths['STAGE_MODEL_DIR']
XGB_MODEL_DIR = stage_paths['XGB_MODEL_DIR']
NN_MODEL_DIR = stage_paths['NN_MODEL_DIR']

ensure_directories(
    DATA_CACHE_DIR,
    TUNING_DIR,
    STAGE_TUNING_DIR,
    XGB_TUNING_DIR,
    NN_TUNING_DIR,
    MODELS_DIR,
    STAGE_MODEL_DIR,
    XGB_MODEL_DIR,
    NN_MODEL_DIR,
    FIGURES_DIR,
)

LOAD_SAVED_TUNING = True
RUN_XGB_TUNING = False
RUN_NN_TUNING = False
RESUME_XGB_TUNING = True
RESUME_NN_TUNING = True

XGB_N_TRIALS = 2000
NN_N_TRIALS = 200

RELOAD_DATA_CACHE = RUNTIME_DEFAULTS['RELOAD_DATA_CACHE']
MANUAL_REVIEW = MANUAL_REVIEW_DEFAULT
SEED = DEFAULT_RANDOM_SEED


def run_keras_tuner(
    max_trials,
    project_name,
    X_train,
    y_train,
    X_val,
    y_val,
    executions_per_trial=1,
    overwrite=False,
    hp_bs=64,
    artifact_dir=None,
    run_search=True,
    load_saved=True,
    resume_search=True,
    artifact_prefix='',
):
    def build_model_for_tuner(hp):
        return build_binary_classifier(
            input_dim=X_train.shape[1],
            units=hp.Int('units', **SEARCH_SPACE['units']),
            layers=hp.Int('layers', **SEARCH_SPACE['layers']),
            activation=hp.Choice('activation', SEARCH_SPACE['activation']),
            optimizer=hp.Choice('optimizer', SEARCH_SPACE['optimizer']),
            lr=hp.Choice('lr', SEARCH_SPACE['lr']),
            dropout=hp.Float('dropout', **SEARCH_SPACE['dropout']),
            l2_strength=hp.Choice('l2_strength', SEARCH_SPACE['l2_strength']),
        )

    early_stop = KerasEarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=5,
        restore_best_weights=True,
        min_delta=1e-4,
    )

    return shared_run_keras_tuner(
        kt_module=kt,
        build_model=build_model_for_tuner,
        max_trials=max_trials,
        project_name=project_name,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        early_stopping_callback=early_stop,
        seed=SEED,
        artifact_dir=artifact_dir or NN_TUNING_DIR,
        executions_per_trial=executions_per_trial,
        overwrite=overwrite,
        hp_bs=hp_bs,
        run_search=run_search,
        load_saved=load_saved,
        resume_search=resume_search,
        artifact_prefix=artifact_prefix,
    )


def run_optuna_xgb(
    n_trials,
    study_name,
    X_train,
    y_train,
    X_val,
    y_val,
    seed=42,
    overwrite=False,
    n_estimators=3000,
    early_stopping_rounds=30,
    n_jobs=-1,
    artifact_dir=None,
    run_search=True,
    load_saved=True,
    resume_search=True,
):
    return shared_run_optuna_xgb(
        study_name=study_name,
        search_space=XGB_SEARCH_SPACE,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        artifact_dir=artifact_dir or XGB_TUNING_DIR,
        model_dir=XGB_MODEL_DIR,
        seed=seed,
        overwrite=overwrite,
        n_trials=n_trials,
        n_estimators=n_estimators,
        early_stopping_rounds=early_stopping_rounds,
        n_jobs=n_jobs,
        run_search=run_search,
        load_saved=load_saved,
        resume_search=resume_search,
        early_stopping_callback_factory=lambda rounds: XgbEarlyStopping(
            rounds=rounds,
            metric_name='auc',
            maximize=True,
        ),
    )

CACHE_PATH = DATA_CACHE_DIR / 'stage_3_data.h5'
hdf_key = 'stage_3_data'

if not RELOAD_DATA_CACHE and CACHE_PATH.exists():
    try:
        stage_3_data = pd.read_hdf(CACHE_PATH, hdf_key)
        print(f'Loaded cached Stage 3 data from {CACHE_PATH}')
    except (OSError, ValueError, KeyError) as e:
        print(f'Failed to load cached Stage 3 data, reloading from CSV: {e}')
        stage_3_data = pd.read_csv(STAGE_DATA_PATH)
        stage_3_data.to_hdf(CACHE_PATH, key=hdf_key, mode='w')
        print(f'Rebuilt cache at {CACHE_PATH}')
else:
    stage_3_data = pd.read_csv(STAGE_DATA_PATH)
    stage_3_data.to_hdf(CACHE_PATH, key=hdf_key, mode='w')
    print(f'Loaded Stage 3 public dataset and refreshed cache at {CACHE_PATH}')

stage_3_data_unmodified = stage_3_data.copy()


**Stage 3: Pre-processing instructions**

- Remove any columns not useful in the analysis (LearnerCode).
- Remove columns with categorical features with high cardinality (use >200 unique values, as a guideline for this data set).
- Remove columns with >50% data missing.
- Perform ordinal encoding for ordinal data.
- Perform one-hot encoding for all other categorical data.
- Choose how to engage with rows that have missing values, which can be done in one of two ways for this project:
  *   Impute the rows with appropriate values.
  *   Remove rows with missing values but ONLY in cases where rows with missing values are minimal: <2% of the overall data.






In [ ]:
stage_3_data.head()


## Data Inspection

First we'll review the data and perform any data cleanse or pre-processing.

### Comparison with Stage 2 Data

Stage 3 data is known to be the original Stage 2 dataset with the addition of attendance metrics, academic metrics, including passed and failed modules.

EDA and feature engineering have already been completed on stage_1_data, and stage_2_data so we will verify the following:

- Columns that do not match those in the Stage 2 dataset
- Any differences in index-aligned rows for the columns common to both datasets

If there are no differences between the original Stage 2 columns and the corresponding columns in Stage 3, then EDA can be focused on the newly added columns only.

In [ ]:
# determine the number of rows that differ between the two datasets for the 
# common columns
common_cols = stage_2_data_unmodified.columns.intersection(
    stage_3_data_unmodified.columns
)

# select only the common columns from both datasets to compare the rows and
# identify any differences in the data for those columns
a = stage_2_data_unmodified[common_cols]
b = stage_3_data_unmodified[common_cols]

# create a boolean mask to identify rows that differ between the two datasets
# for the common columns, accounting for potential NaN values by treating rows
# as the same if they are both NaN in a column
diff_mask = ~((a == b) | (a.isna() & b.isna())).all(axis=1)

# sum the boolean mask to get the total count of rows that differ between the 
# two datasets for the common columns
diff_count = diff_mask.sum()

print("Rows that differ:", diff_count)

new_cols_stage2 = list(stage_3_data_unmodified.columns.difference(common_cols))

print("New Columns in Stage 3:\n", new_cols_stage2)

# copy the new columns to a new dataframe for easier analysis and to understand
# the additional information available in the stage 2 dataset that was not present
# in stage 1, which may be useful for further analysis or modelling.
stage_3_new_cols = stage_3_data_unmodified[new_cols_stage2]

# View the metadata with the info function.
print("Datatypes: \n" + str(stage_3_new_cols.info()) + "\n")


There are three new columns: 

- AssessedModules (float64)
- FailedModules   (float64)
- PassedModules   (float64)

All other columns match stage 2 data exactly, therefore we will focus EDA on the two new columns only and then add them to the stage 2 data post encoding if they are kept.

### Data Quality Checks

We will perform basic data quality checks, view the dataframe metadata to determine assigned datatypes, and determine the size of the dataset.

Data quality checks (numeric fields):

- Missing data checks
- Count of unique values
- Count of null rows

In [ ]:
# Check for null values
print(
    "\nData quality check (1) detect null values in columns:\n"
    + str(
        stage_3_new_cols.isna()
        .sum()
        .to_frame("null_count")
        .assign(null_pct=lambda x: (x / len(stage_1_data) * 100).round(2))
        .sort_values(by="null_count", ascending=False)
    )
    + "\n"
)


# count unique values in each column and display count & percentage
print(
    "\nData quality check (2) unique values in each column:\n"
    + str(
        stage_3_new_cols.nunique()
        .to_frame("distinct_count")
        .assign(distinct_pct=lambda x: (x / len(stage_1_data) * 100).round(2))
        .sort_values(by="distinct_count", ascending=True)
    )
    + "\n"
)

# count the number of rows where the both new columns are null to understand the
# extent of missing data in the new columns and to assess whether it may be a
# significant issue for analysis or modeling.
missing_all = stage_3_new_cols.isna().all(axis=1).sum()
print(
    f"Number of rows where all new columns are null: {missing_all} "
    f"({(missing_all / len(stage_3_data) * 100):.2f}%)"
)



**Initial Data Quality Check**

---

**Data summary**

- 25,059 records
- All columns from Stage 2 are present, with three additional fields:
  - `AssessedModules`
  - `FailedModules`
  - `PassedModules`

**Null values in the new fields**

All three new assessment fields have 2,231 null values (8.9% of the dataset), and the nulls occur on the same rows. This strongly suggests missing assessment data rather than true zero values.

**Missing-data approach**

Because these rows represent a relatively small share of the dataset, we will test simple imputation and preserve the missingness signal explicitly through a dedicated indicator feature.

We will examine the feature distributions and the relationship between missingness and the target variable before finalising the preprocessing approach.

---

### Missing Data Analysis


In [ ]:
##investigate null and zero values in the numeric columns to see if there are any
# patterns in the dropout rate for those with null or zero values compared to
# those with positive values.

stage_3_new_cols = stage_3_new_cols.copy()
stage_3_new_cols["dropout"] = stage_1_data["dropout"]

for col in ["AssessedModules", "FailedModules", "PassedModules"]:
    summary = (
        stage_3_new_cols.assign(
            category=lambda df: np.select(
                [df[col].isna(), df[col] == 0],
                ["NULL", "ZERO"],
                default="PRESENT",
            )
        )
        .groupby("category")["dropout"]
        .agg(count="size", dropout_rate="mean")
        .reset_index()
    )

    plt.figure(figsize=(6, 4))

    ax = sns.barplot(
        data=summary,
        x="category",
        y="dropout_rate",
        order=["NULL", "ZERO", "PRESENT"],
    )

    ax.set_title(f"Dropout rate by {col}")
    ax.set_ylim(0, 1)

    for i, row in summary.iterrows():
        ax.text(
            i, row["dropout_rate"] + 0.02, f"n={row['count']}", ha="center"
        )

    plt.tight_layout()
    plt.show()


### Exploratory Data Analysis

Generate descriptive statistics and visualise the data to explore patterns, distributions, and trends in the data.

**Descriptive Statistics**

We will now generate descriptive statistics (including percentile values for all numeric columns in the dataset), and document any insights this provides about the data.

In [ ]:
# Use df.describe to generate descriptive stats of new columns to understand the
# distribution of values, central tendency, and variability in the new columns,
# which can inform further analysis or modelling decisions.

stage_3_new_cols.describe(
    include="all", percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
).fillna(0).T.sort_index()


**Visualise the data**

Visualise the numeric data with box-plots and histograms to further understand the data patterns, distributions, and trends.


In [ ]:
# choose columns to plot - exclude categorical columns
cols = ["AssessedModules", "FailedModules", "PassedModules"]

# plot histograms and boxplots for the numeric columns to visualise their 
# distributions and check for outliers

plot_distributions_and_qq(stage_3_new_cols, cols)


**Interpretation**

---

**AssessedModules**

- The number of assessed modules is tightly distributed, with a median of 6 and an interquartile range of 4 to 7.
- The minimum value is 1 and the maximum is 12, indicating a bounded range.
- The mean (6.09) is close to the median, suggesting a roughly symmetric distribution with limited skew.
- The histogram confirms that most students are assessed on between 4 and 8 modules, with relatively few extreme values.

**PassedModules**

- PassedModules shows a similar distribution to AssessedModules, with a median of 6 and a mean of 5.58.
- The lower quartile is 4 and the upper quartile is 7, indicating that most students pass the majority of assessed modules.
- The minimum of 0 and maximum of 11 suggest that some students fail all modules, while others pass nearly all.

**FailedModules**

- FailedModules is highly right-skewed, with a median of 0 and the 75th percentile also equal to 0.
- This indicates that most students do not fail any modules.
- The mean (0.51) is higher than the median due to a small number of students with multiple failed modules.
- The histogram and boxplot confirm strong zero inflation with a small number of extreme values.

These characteristics suggest that failure-related features are likely to contain strong predictive signal for dropout.

**Missing Data**

Module count values contain a substantial number of missing records, and missingness shows a very strong association with dropout.

Records where module counts are missing exhibit extremely high dropout rates compared to records where module data is present. This indicates that missingness itself carries predictive information and should not be ignored.

Removing these rows would discard a large proportion of the dataset and remove informative signal, therefore the rows will be retained and missingness will be encoded explicitly.

---

## Pre-Processing

### Imputation

To preserve the missing-value signal, a single feature module_values_missing will be created based on the AssessedModules / PassedModules / FailedModules values.

This feature will be defined as follows:

- All values null → 1  
- Values present → 0  

As the module fields are missing simultaneously, a single missingness indicator is sufficient.

Null values in the module count fields will be imputed with zero to allow model training while preserving the missingness signal.

The resulting feature interpretation is shown below:

| module_values_missing | FailedModules | Meaning   |
|----------------------|--------------|-----------|
| 0                    | 2            | real      |
| 0                    | 0            | real zero |
| 1                    | 0            | missing   |


### Encoding

***Not required***  
The new columns are numeric count values or binary indicators, therefore no additional encoding is required.

As these features are being added to the previously processed Stage 2 data, the new columns can be appended to the Stage 2 feature set after encoding but before scaling.


### Scaling

The module count features are bounded integer values with moderate range and do not require transformation.

However, scaling is required for neural network training to ensure comparable feature magnitudes.

StandardScaler will be retained to maintain consistency with previous stages and allow direct comparison between models.  
The combined Stage 1, Stage 2, and module features must therefore be scaled prior to neural network training and evaluation.

In [ ]:
# create new module_values_missing column to indicate rows where both absence 
# count columns are null
stage_3_new_cols = stage_3_new_cols.assign(
    module_values_missing=lambda df: (
        df[["AssessedModules", "FailedModules", "PassedModules"]]
        .isna()
        .all(axis=1)
        .astype(int)
    )
)

# fill null values in the absence count columns with 0 placeholder
stage_3_new_cols["AssessedModules"] = stage_3_new_cols["FailedModules"].fillna(
    0
)
stage_3_new_cols["FailedModules"] = stage_3_new_cols["FailedModules"].fillna(0)
stage_3_new_cols["PassedModules"] = stage_3_new_cols["PassedModules"].fillna(0)

# add the new columns back to the post-encoding stage 2 dataset
stage_3_data_encoded = pd.concat(
    [
        stage_2_data_encoded,
        stage_3_new_cols[
            [
                "AssessedModules",
                "FailedModules",
                "PassedModules",
                "module_values_missing",
            ]
        ],
    ],
    axis=1,
)

print("Stage 3 dataset shape:", stage_3_data_encoded.shape)


## Train / validation / test split

We will create a train, test, and validation data set. Training set split 80-20, with 10% of the training set used as validation.

As the target is imbalanced we will use stratified split.

We also create scaled versions of the feature matrices for neural network training, as neural networks rely on gradient-based optimisation and are sensitive to differences in feature scale

In [ ]:
stage_3_data_encoded.head()


In [ ]:
# create train, validation, and test splits for the stage 2 dataset using the 
# same function as before to ensure consistency in how we split the data for 
# modeling and evaluation.
(
    X_train,
    X_train_s,
    X_val,
    X_val_s,
    X_test,
    X_test_s,
    y_train,
    y_val,
    y_test,
    scaler,
) = create_train_val_test_split_and_scale(
    stage_3_data_encoded, stratify=True, seed=SEED
)


## Stage 3 XGBoost Model

### Baseline Model

A new XGBoost model will be trained and evaluated on the updated dataset using the best-performing hyperparameters identified during Stage 2. Model performance will then be compared with the Stage 2 results.

A new model is required because the feature set has changed, resulting in different input dimensionality.

In [ ]:

previous_stage_xgb_params = load_saved_best_params(PREV_XGB_TUNING_DIR / TUNING_ARTIFACT_FILENAMES['best_params'])
if previous_stage_xgb_params is None:
    previous_stage_xgb_params = {
        'learning_rate': 0.119,
        'max_depth': 8,
        'min_child_weight': 1.15,
        'gamma': 0.27,
        'subsample': 0.754,
        'colsample_bytree': 0.601,
        'reg_alpha': 0.0052,
        'reg_lambda': 4.74,
    }
    print(
        'No saved Stage 2 XGBoost params were found. Falling back to '
        'the committed Stage 2 baseline parameter record.'
    )

best_xgb_params = previous_stage_xgb_params
print('Baseline XGBoost hyperparameters carried forward from Stage 2:')
print(best_xgb_params)

set_seed(SEED)
xgb_model = xgb.XGBClassifier(
    **best_xgb_params,
    random_state=SEED,
    eval_metric='auc',
    n_jobs=N_JOBS,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]


### Baseline Performance Metrics

We will now evaluate the model against the test set using the following performance indicators:
 - Accuracy
 - Precision
 - Recall
 - AUC

A confusion matrix, ROC and Precision-Recall Curves will also be plotted.

These will be compared against model performance from the model when used with stage 2 data

In [ ]:
m_id = 9  # unique identifier for this model in the results dictionary
m_ids = [6, m_id]  # list of model m_ids to compare in the plots

# evaluate the model using the evaluate_and_store_model function defined earlier,
# which computes performance metrics and stores them in the results dictionary
results = evaluate_and_store_model(
    results,
    m_id=m_id,
    model_name="Baseline",
    model_type="XG Boost",
    stage="Stage 3",
    y_true=y_test,
    y_pred=y_pred_xgb,
    y_prob=y_prob_xgb,
    hyperparameters=best_xgb_params,
    metadata={"threshold": 0.5},
)

# print the evaluation metrics for the XGBoost baseline model in a consistent 
# format using the print_model_metrics function defined earlier
print_model_metrics(results, m_ids[0])
print("\nStage 2 XGBoost Tuned Confusion Matrix:")
plot_model_confusion_matrix(results, m_ids[0])

print_model_metrics(results, m_id)
print("\nStage 3 XGBoost Baseline Confusion Matrix:")
plot_model_confusion_matrix(results, m_id)

models_to_plot = build_models_to_plot(results, m_ids)
plot_roc_and_pr_curves(models_to_plot, figsize=(14, 7))


Comparison of XGBoost Stage 2 (Tuned) vs XGBoost Stage 3 (Baseline)

Model performance improves substantially when the Stage 2 tuned model is compared with the model trained on the Stage 3 dataset. AUC increases from 0.9340 to 0.9987, indicating that the model is able to almost perfectly separate students who drop out from those who complete. This is visible in the ROC curve, where the Stage 3 curve lies very close to the top-left corner, showing near-perfect discrimination across all thresholds.

| Metric      | Stage 3 | Stage 2 | Change   | Business Interpretation                                                                                  |
| ----------- | ------- | ------- | -------- | -------------------------------------------------------------------------------------------------------- |
| TP          | 701     | 477     | ↑ better | Many more students who will drop out are correctly identified, improving the ability to intervene early. |
| FN          | 50      | 274     | ↓ better | Very few at-risk students are missed, meaning support can be targeted more reliably.                     |
| FP          | 19      | 143     | ↓ better | Far fewer students who will complete are incorrectly flagged, reducing unnecessary intervention effort.  |
| TN          | 4242    | 4118    | ↑ better | Almost all students who will complete are correctly recognised.                                          |
| Recall      | 0.93    | 0.64    | ↑ better | The model detects most dropout cases, making it highly effective as an early warning system.             |
| Specificity | 0.996   | 0.966   | ↑ better | Nearly all non-dropout students are correctly classified, keeping intervention focused on genuine risk.  |


The Precision–Recall curve also shows a very large improvement, with precision remaining close to 1.0 across most recall values. This indicates that the increase in recall is not achieved by increasing false positives, but by the model having access to much stronger predictive information.

The magnitude of the improvement is far larger than any change observed through hyperparameter tuning, indicating that the performance gain is primarily due to the additional features introduced in the Stage 3 dataset rather than changes to the model configuration.

The large increase in performance suggests that the Stage 3 variables contain information that is highly predictive of the final outcome, allowing the model to distinguish almost perfectly between students who complete and those who drop out.

Overall, the results show that the additional Stage 3 features provide extremely strong predictive signal, leading to near-perfect classification performance.


### Hyperparameter Tuning

Hyperparameters will now be tuned using the same search space previously used for the Stage 1 and 2 model, as summarised below.

Optuna with Bayesian optimisation will be used again to maintain consistency and comparability with the Stage 1 and 2 tuning process.

In [ ]:
set_seed(SEED)

XGB_SEARCH_SPACE = {
    'learning_rate': {'low': 1e-2, 'high': 2e-1, 'log': True},
    'max_depth': {'low': 3, 'high': 8},
    'min_child_weight': {'low': 1.0, 'high': 12.0, 'log': True},
    'gamma': {'low': 0.0, 'high': 5.0},
    'subsample': {'low': 0.60, 'high': 1.00},
    'colsample_bytree': {'low': 0.60, 'high': 1.00},
    'reg_alpha': {'low': 1e-8, 'high': 10.0, 'log': True},
    'reg_lambda': {'low': 1e-2, 'high': 50.0, 'log': True},
}

optuna.logging.set_verbosity(optuna.logging.WARNING)

best_xgb_model, best_xgb_params, best_val_auc, study = run_optuna_xgb(
    n_trials=XGB_N_TRIALS,
    study_name='xgb_stage_3_tabular_auc',
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    seed=SEED,
    n_jobs=N_JOBS,
    overwrite=False,
    artifact_dir=XGB_TUNING_DIR,
    run_search=RUN_XGB_TUNING,
    load_saved=LOAD_SAVED_TUNING,
    resume_search=RESUME_XGB_TUNING,
)

if best_xgb_model is None:
    print('Falling back to the Stage 3 baseline XGBoost model because no stage-specific tuning artefacts are available.')
    best_xgb_model = xgb_model
    best_xgb_params = previous_stage_xgb_params
    best_val_auc = None

print('Best validation AUC:', best_val_auc)
print('Best params:', best_xgb_params)


In [ ]:

trials_path = XGB_TUNING_DIR / TUNING_ARTIFACT_FILENAMES['trials']

if study is not None:
    df_trials = study.trials_dataframe()
elif trials_path.exists():
    df_trials = pd.read_csv(trials_path)
    print(f'Loaded saved XGBoost trial history from {trials_path}')
else:
    df_trials = pd.DataFrame([
        {'number': 0, 'value': best_val_auc, **{f'params_{k}': v for k, v in best_xgb_params.items()}},
    ])
    print('No Stage 3 XGBoost trial history is available. Using the saved best-parameter record only.')

available_hyperparams = [col for col in xgb_hyperparams if col in df_trials.columns]
xgb_results_df_combined = pd.concat(
    [
        xgb_results_df_combined,
        pd.DataFrame(
            {
                'trial_number': df_trials['number'],
                'val_auc': df_trials['value'],
                **{col: df_trials[col] for col in available_hyperparams},
                'Stage': 'Stage 3',
            }
        ),
    ],
    ignore_index=True,
)

display(df_trials.sort_values('value', ascending=False).head(10))


### Tuned Model

Hyperparameters for the best performing model (trial 1798) are shown below and compared to the baseline model:

| Hyperparameter       | Stage 2 Tuned (Stage 3 Baseline) | Stage 3 Tuned  | Interpretation                                                                              |
| -------------------- | -------------------------------- | -------------- | ------------------------------------------------------------------------------------------- |
| **learning_rate**    | 0.119                            | **0.200**      | Higher learning rate in Stage 3 allows faster convergence due to stronger predictive signal |
| **max_depth**        | 8                                | **8**          | Tree depth unchanged, suggesting the previous structure was already sufficient              |
| **min_child_weight** | 1.15                             | **1.18**       | Similar leaf constraints, indicating stable splits already achieved                         |
| **gamma**            | 0.27                             | **0.67**       | Higher split penalty reduces unnecessary splits when features are highly predictive         |
| **subsample**        | 0.754                            | **0.932**      | Less subsampling in Stage 3 allows the model to use more data per tree                      |
| **colsample_bytree** | 0.601                            | **0.612**      | Feature sampling similar, indicating no need for strong feature regularisation              |
| **reg_alpha (L1)**   | 0.0052                           | **0.00000002** | L1 regularisation almost removed, suggesting sparsity control is no longer required         |
| **reg_lambda (L2)**  | 4.74                             | **0.064**      | Much lower L2 regularisation, suggesting the Stage 3 features provide cleaner separation - the model does not need strong regularisation because the features already separate the classes well       |
| **n_estimators**     | Early stopping                   | Early stopping | Boosting rounds determined automatically using validation AUC                               |


Only moderate changes to the hyperparameters are observed between the Stage 2 tuned configuration and the Stage 3 tuned model. Tree depth remains unchanged, indicating that the existing model structure was already sufficient to capture the relationships in the data.

The most notable differences are the increase in learning rate and the large reduction in L2 regularisation. The lower regularisation suggests that the Stage 3 features provide a much clearer separation between dropout and non-dropout cases, allowing the model to fit the data without requiring strong constraints to prevent overfitting.

Overall, the relatively small changes in hyperparameters confirm that the performance improvement in Stage 3 is primarily driven by the additional engineered features rather than by extensive hyperparameter optimisation.


### Tuned Model Evaluation

**Performance Metrics**

We will now evaluate the model against the same performance parameters used to evaluate the stage 2 model.

In [ ]:
# make predictions on the test set using the fitted XGBoost model
y_pred_xgb = best_xgb_model.predict(X_test)
y_prob_xgb = best_xgb_model.predict_proba(X_test)[:, 1]

m_id = 10  # unique identifier for this model in the results dictionary
m_ids = [9, m_id]  # list of model m_ids to compare in the plots

# store the results in the results dictionary for the tuned XGBoost model
results = evaluate_and_store_model(
    results,
    m_id=m_id,
    model_name="Tuned",
    model_type="XG Boost",
    stage="Stage 3",
    y_true=y_test,
    y_pred=y_pred_xgb,
    y_prob=y_prob_xgb,
    hyperparameters=best_xgb_params,
    metadata={"threshold": 0.5},
)

# print the evaluation metrics and plot the confusion matrix for the tuned XGBoost model

print_model_metrics(results, m_id - 1)
print("\nStage 3 XGBoost Baseline Confusion Matrix:")
plot_model_confusion_matrix(results, m_id - 1)

print_model_metrics(results, m_id)
print("\nStage 3 XGBoost Tuned Confusion Matrix:")
plot_model_confusion_matrix(results, m_id)

models_to_plot = build_models_to_plot(results, m_ids)
plot_roc_and_pr_curves(models_to_plot, figsize=(14, 7))


### Stage 3 vs Stage 2 Comparison

The tuned Stage 3 neural network performs slightly better than the baseline model, but the difference is small relative to the already very high performance achieved using the Stage 3 dataset.

| Metric      | Baseline | Tuned      |
| ----------- | -------- | ---------- |
| Accuracy    | 0.9769   | **0.9862** |
| Precision   | 0.9441   | **0.9830** |
| Recall      | 0.8988   | **0.9241** |
| Specificity | 0.9906   | **0.9972** |
| AUC         | 0.9958   | **0.9987** |


Hyperparameter tuning on the Stage 3 dataset produces only a modest improvement over the baseline configuration. Validation AUC during tuning was already very close to 1.0, and the final test AUC of 0.9987 confirms that the model is able to almost perfectly separate dropout and completion cases even without extensive optimisation.

The ROC and Precision–Recall curves for the baseline and tuned models are very similar, with the tuned model lying slightly above the baseline across most thresholds. This indicates that tuning improves overall ranking performance but does not fundamentally change the classification behaviour.

The confusion matrices show that the tuned model produces fewer false negatives and fewer false positives, increasing both recall and specificity. In particular, false negatives decrease from 76 to 57, meaning more dropout cases are correctly identified, while false positives decrease from 40 to 12, meaning fewer students are incorrectly flagged as at risk.

As with the XGBoost results, the improvement from tuning is much smaller than the improvement observed when moving from Stage 2 to Stage 3. This indicates that the major performance gain comes from the additional Stage 3 features rather than from hyperparameter optimisation.

Once the highly informative Stage 3 variables are introduced, model performance becomes very high and relatively insensitive to hyperparameter choice, with tuning providing only small additional gains.

### Feature Importance

We will now review feature importance with the addition of the new features.

Both feature importance and SHAP plots will be created so that we can compare how well they align and use SHAP to better understand magnitude and direction of feature influence.

In [ ]:
# plot the feature importance for the XGBoost model using the feature_importances_ attribute of the fitted model, and display the top 20 most important features
feature_importance = pd.Series(
    best_xgb_model.feature_importances_, index=X_train.columns
).sort_values()
plt.figure(figsize=(10, 35))
feature_importance.plot.barh()
plt.show()

plt.figure(figsize=(10, 10))
feature_importance.iloc[-20:].plot.barh()
plt.show()


In [ ]:
# For the SHAP values, we will use the TreeExplainer which is optimized for tree-based models like XGBoost.
# We will compute the SHAP values for the test set and plot a summary plot to visualize the overall feature importance and the distribution of SHAP values for the top features.

explainer = shap.TreeExplainer(best_xgb_model)
shap_values = explainer.shap_values(X_test)

# plot the SHAP summary plot (beeswarm plot) to show the impact of features on the model's predictions across the test set
# limit to top 40 features for better visualization
shap.plots.violin(shap_values, X_test, max_display=40, show=False)

fig = plt.gcf()
ax = plt.gca()

# Reduce font size for feature names (y-axis) and SHAP values (x-axis)
ax.tick_params(axis="both", which="major", labelsize=10)

# Reduce font size for labels
ax.xaxis.label.set_size(14)
ax.yaxis.label.set_size(14)

plt.show()


In [ ]:
# Finally, we can also plot the grouped feature importance for both the XGBoost
# feature importance and SHAP values side by side for comparison.
# The features are grouped into categories based on their prefixes

plot_grouped_feature_importance(feature_importance, shap_values, X_val)


**Interpretation**

SHAP importance indicates that the new assessment-related features provide the strongest contribution to model predictions by a large margin. This suggests that assessment performance is the most informative predictor of dropout in the Stage 3 dataset, and explains the substantial improvement in model performance following the addition of these variables. 

The difference between the two importance measures highlights that assessment features have both strong per-sample influence and strong global predictive power across the dataset. Unlike the absence features in Stage 2, which affected a smaller number of students, the assessment variables provide a clear and consistent signal for most observations.

This explains why the Stage 3 model achieves near-perfect performance, as the additional features allow the model to distinguish between students who drop out and those who complete with very high confidence.

## Stage 3 Neural Network Model

### Baseline Model

A new Neural Network model will be trained and evaluated on the updated dataset using the best-performing hyperparameters identified during Stage 2. Model performance will then be compared with the Stage 2 results.

A new model is required because the feature set has changed, resulting in different input dimensionality.

In [ ]:

previous_stage_nn_params = load_saved_best_params(PREV_NN_TUNING_DIR / TUNING_ARTIFACT_FILENAMES['best_params'])
if previous_stage_nn_params is None:
    previous_stage_nn_params = {
        'units': 160,
        'layers': 3,
        'activation': 'relu',
        'optimizer': 'adam',
        'lr': 0.001,
        'dropout': 0.3,
        'l2_strength': 0.0,
    }
    print(
        'No saved Stage 2 neural-network params were found. Falling '
        'back to the committed Stage 2 baseline parameter record.'
    )

best_nn_params = previous_stage_nn_params
print('Baseline neural-network hyperparameters carried forward from Stage 2:')
print(best_nn_params)

retrain_tuned = False
loaded = False

tf.keras.backend.clear_session()
set_seed(SEED)

model_path = NN_MODEL_DIR / 'baseline_from_stage_2_tuned.keras'
model = build_binary_classifier(input_dim=X_train_s.shape[1], **best_nn_params)

try:
    if not retrain_tuned:
        model, loaded = load_model_weights(model, model_path)
        if loaded:
            print(f'Model loaded from {model_path}')
except Exception as e:
    print(f'Error loading model from {model_path}: {e}')
    print('Proceeding to train the model and save weights for future use.')
    loaded = False

if not loaded:
    early_stop = KerasEarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=5,
        restore_best_weights=True,
        min_delta=1e-4,
    )
    history = model.fit(
        X_train_s,
        y_train,
        validation_data=(X_val_s, y_val),
        epochs=50,
        batch_size=64,
        callbacks=[early_stop],
        verbose=1,
    )
    model.save(model_path)


### Baseline Performance Metrics

We will now evaluate the model against the test set using the following performance indicators:
 - Accuracy
 - Precision
 - Recall
 - AUC

A confusion matrix, ROC and Precision-Recall Curves will also be plotted.

These will be compared against model performance from the model when used with stage 1 data.

In [ ]:

m_id = 11
m_ids = [8, m_id]

y_prob = model.predict(X_test_s, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

results = evaluate_and_store_model(
    results,
    m_id=m_id,
    model_name='Baseline',
    model_type='Neural Network',
    stage='Stage 3',
    y_true=y_test,
    y_pred=y_pred,
    y_prob=y_prob,
    hyperparameters=best_nn_params,
    metadata={'threshold': 0.5},
)

print_model_metrics(results, m_ids[0])
print()
print('Stage 2 Neural Network Tuned Confusion Matrix:')
plot_model_confusion_matrix(results, m_ids[0])

print_model_metrics(results, m_id)
print()
print('Stage 3 Neural Network Baseline Confusion Matrix:')
plot_model_confusion_matrix(results, m_id)

models_to_plot = build_models_to_plot(results, m_ids)
plot_roc_and_pr_curves(models_to_plot, figsize=(14, 7))



**Comparison of Neural Network Stage 2 (Tuned) vs Neural Network Stage 3 (Baseline)**

---

AUC increases from 0.9220 to 0.9958, indicating a very large improvement in the model’s ability to separate students who drop out from those who complete across all decision thresholds. This is reflected in the ROC curve, where the Stage 3 curve lies much closer to the top-left corner than the Stage 2 curve, resulting in near-perfect discrimination.


| Metric      | Stage 3 | Stage 2 | Change   | Business Interpretation                                                                                                    |
| ----------- | ------- | ------- | -------- | -------------------------------------------------------------------------------------------------------------------------- |
| TP          | 675     | 436     | ↑ better | Many more students who will drop out are correctly identified, greatly improving the ability to target early intervention. |
| FN          | 76      | 315     | ↓ better | Far fewer at-risk students are missed, meaning support can be applied much more reliably.                                  |
| FP          | 40      | 139     | ↓ better | Fewer students who will complete are incorrectly flagged as at risk, reducing unnecessary intervention effort.             |
| TN          | 4221    | 4122    | ↑ better | More students who will complete are correctly identified, improving overall classification reliability.                    |
| Recall      | 0.899   | 0.581   | ↑ better | A much higher proportion of actual dropouts are detected, making the model far more effective as an early warning system.  |
| Specificity | 0.991   | 0.967   | ↑ better | Almost all non-dropout students are correctly classified, keeping intervention focused on genuine risk cases.              |


As with the XGBoost model, when the Stage 2 tuned architecture is evaluated using the Stage 3 dataset, performance improves dramatically compared with evaluation on the Stage 2 data.

The magnitude of the improvement is far larger than any change observed through hyperparameter tuning, confirming that the performance gain is primarily due to the additional Stage 3 features rather than changes to the neural network architecture.

Overall, the Stage 3 variables provide extremely strong predictive signal, allowing the neural network to achieve near-perfect classification performance, similar to the behaviour observed with the XGBoost model.

---


### Hyperparameter Tuning

Saved tuning artefacts are loaded by default so this notebook can be rerun quickly. Stage 3 neural-network tuning is computationally expensive, so `RUN_NN_TUNING` is set to `False` by default. Set it to `True` only if you want to rerun the RandomSearch workflow and refresh the saved files under `tuning/stage_3/neural_network/`.

The Stage 3 dataset introduces a small number of additional features, but the overall structure of the problem remains similar to Stage 2. For consistency, the neural-network tuning workflow continues to use the refined search space from the earlier stages.


In [ ]:
tf.keras.backend.clear_session()
set_seed(SEED)

SEARCH_SPACE = {
    'units': {'min_value': 32, 'max_value': 192, 'step': 16},
    'layers': {'min_value': 1, 'max_value': 4},
    'dropout': {'min_value': 0.0, 'max_value': 0.4, 'step': 0.1},
    'activation': ['relu', 'tanh'],
    'optimizer': ['adam'],
    'lr': [3e-4, 1e-3],
    'l2_strength': [0.0, 1e-6, 1e-5, 1e-4],
}

best_hp_values, tuner = run_keras_tuner(
    max_trials=NN_N_TRIALS,
    project_name='nn_tabular_auc_refined_stage_3',
    X_train=X_train_s,
    y_train=y_train,
    X_val=X_val_s,
    y_val=y_val,
    overwrite=False,
    artifact_dir=NN_TUNING_DIR,
    run_search=RUN_NN_TUNING,
    load_saved=LOAD_SAVED_TUNING,
    resume_search=RESUME_NN_TUNING,
)


In [ ]:

nn_trials_path = NN_TUNING_DIR / TUNING_ARTIFACT_FILENAMES['trials']
saved_nn_record = best_params_record(NN_TUNING_DIR / TUNING_ARTIFACT_FILENAMES['best_params'])

if tuner is not None:
    results_df = tuner_trials_to_dataframe(tuner)
elif nn_trials_path.exists():
    results_df = pd.read_csv(nn_trials_path).sort_values('val_auc', ascending=False)
    print(f'Loaded saved Stage 3 NN tuning trials from {nn_trials_path}')
elif saved_nn_record:
    results_df = pd.DataFrame([{
        **saved_nn_record.get('best_params', {}),
        'val_auc': saved_nn_record.get('best_val_auc'),
        'trial_id': 'saved_best_params',
    }])
    print('No Stage 3 NN trial history is available. Using the saved best-parameter record only.')
else:
    results_df = pd.DataFrame()
    print('No Stage 3 NN tuning artefacts are available. Set RUN_NN_TUNING = True to generate them.')

if not results_df.empty:
    display(HTML('<div style="max-height:330px; overflow:auto;">' + results_df.to_html() + '</div>'))
    results_df_combined = pd.concat([results_df_combined, results_df.assign(source='stage3_refined')], ignore_index=True)
else:
    results_df_combined = results_df_combined.copy()


In [ ]:
best_idx = 5

if results_df.empty:
    best_idx = 0

if MANUAL_REVIEW and best_idx == -1:
    raise RuntimeError(
        "Stability optimisation complete.\n"
        "Review results, set SELECTED_CONFIG_IDX in the next cell,\n"
        "then set UPDATE_STABILITY = False and rerun."
    )
elif best_idx == -1:
    best_idx = 0
    print('No configuration index set for final model selection, defaulting to the top available Stage 3 configuration.')


**Refined Hyperparameter Tuning Results**

---

The Stage 3 hyperparameter search results show that validation AUC remains consistently close to 1.0 across a wide range of model configurations, indicating that the additional Stage 3 features provide very strong predictive signal.

Performance differences between the top trials are extremely small, with many configurations achieving almost identical validation AUC values. As a result, model selection is based not only on AUC, but also on model stability, training loss, and architectural simplicity.

Model stability testing across multiple random seeds will not be repeated for Stage 3. The model already achieves near-perfect performance, and previous stability tests in Stage 2 showed very small variation between runs.

Across the top-ranked trials, deeper networks appear more frequently, and the best results consistently use the **tanh** activation function with the **Adam** optimiser. Moderate dropout and small L2 regularisation are also present in most high-performing configurations, suggesting that regularisation is still beneficial even when the predictive signal is strong.

---

### Tuned Model

The final neural network configuration was selected from the top-ranked trials based on validation AUC while also favouring a stable architecture with moderate depth and low validation loss. Trial 053 was chosen because it achieved near-maximum validation AUC while using fewer layers than the deepest models and producing a lower validation loss, reducing the risk of unnecessary complexity.

| Model        | Units | Layers | Activation | Optimizer | LR     | Dropout | L2      | Val AUC |
|------------|-------|--------|-----------|----------|--------|---------|---------|---------|
| Trial 053  | 160   | 2      | tanh      | adam     | 0.0010 | 0.1     | 0.00001 | 0.9993  |


### Tuned Model Evaluation

**Performance Metrics**

We will now evaluate the model against the same performance parameters used to evaluate the stage 2 model.

In [ ]:

tf.keras.backend.clear_session()
set_seed(SEED)

retrain_tuned = False
loaded = False

if results_df.empty:
    config = best_nn_params.copy()
    best_idx = 0
    print('Falling back to the saved Stage 3 NN best-parameter record because no tuning trial history is available.')
else:
    row = results_df.iloc[best_idx]
    config = {
        'units': int(row['units']),
        'layers': int(row['layers']),
        'activation': row['activation'],
        'optimizer': row['optimizer'],
        'lr': float(row['lr']),
        'dropout': float(row['dropout']),
        'l2_strength': float(row['l2_strength']),
    }

model_path = NN_MODEL_DIR / f'tuned_model_config_{best_idx}.keras'
model = build_binary_classifier(input_dim=X_train_s.shape[1], units=config['units'], layers=config['layers'], activation=config['activation'], optimizer=config['optimizer'], lr=config['lr'], dropout=config['dropout'], l2_strength=config['l2_strength'])

try:
    if not retrain_tuned:
        model, loaded = load_model_weights(model, model_path)
        if loaded:
            print(f'Model loaded from {model_path} for config {best_idx}')
except Exception as e:
    print(f'Error loading model from {model_path}: {e}')
    print('Proceeding to train the model and save weights for future use.')
    loaded = False

if not loaded:
    early_stop = KerasEarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True, min_delta=1e-4)
    history = model.fit(X_train_s, y_train, validation_data=(X_val_s, y_val), epochs=50, batch_size=64, callbacks=[early_stop], verbose=1)
    model.save(model_path)


In [ ]:
# display the hyperparameters of the best performing configuration from the refined search for the stage 3 dataset

print(
    "Best performing hyperparameters from the refined search on stage 3 dataset:"
)
display(
    HTML(
        '<div style="max-height:330px; overflow:auto;">'
        + pd.DataFrame([config]).to_html()
        + "</div>"
    )
)


In [ ]:
m_id = 12  # unique identifier for this model in the results dictionary
m_ids = [m_id - 1, m_id]  # list of model m_ids to compare in the plots

# Predict probabilities
y_prob = model.predict(X_test_s, verbose=0).ravel()
# Convert probabilities to binary predictions using a threshold of 0.5
y_pred = (y_prob >= 0.5).astype(int)


results = evaluate_and_store_model(
    results,
    m_id=m_id,
    model_name="Tuned",
    model_type="Neural Network",
    stage="Stage 3",
    y_true=y_test,
    y_pred=y_pred,
    y_prob=y_prob,
    hyperparameters=config,
    metadata={"threshold": 0.5},
)

# print the evaluation metrics for the XGBoost baseline model in a consistent 
# format using the print_model_metrics function defined earlier
print_model_metrics(results, m_ids[0])
print("\nStage 3 Neural Network Baseline Confusion Matrix:")
plot_model_confusion_matrix(results, m_ids[0])


print_model_metrics(results, m_id)
print("\nStage 3 Neural Network Tuned Confusion Matrix:")
plot_model_confusion_matrix(results, m_id)

models_to_plot = build_models_to_plot(
    results,
    m_ids,
)
plot_roc_and_pr_curves(models_to_plot, figsize=(14, 7))



### Stage 3 vs Stage 2 Comparison

As observed with the XGBoost model, the tuned Stage 3 neural network shows only a modest improvement compared with the baseline model when both are evaluated on the Stage 3 dataset. Because the Stage 3 features already provide very strong predictive signal, overall performance is very high even without tuning, and further optimisation results in only small gains.

| Metric      | Baseline | Tuned      |
| ----------- | -------- | ---------- |
| Accuracy    | 0.9769   | **0.9862** |
| Precision   | 0.9441   | **0.9830** |
| Recall      | 0.8988   | **0.9241** |
| Specificity | 0.9906   | **0.9972** |
| AUC         | 0.9958   | **0.9987** |


Hyperparameter tuning on the Stage 3 dataset produces only a limited improvement over the baseline configuration. The ROC curves for the two models are already very close to the top-left corner, and the tuned model lies only slightly above the baseline, indicating that both models achieve near-perfect discrimination.

The confusion matrices show that the tuned model reduces both false negatives and false positives. False negatives decrease from 76 to 57, increasing recall, while false positives decrease from 40 to 12, increasing specificity. This means the tuned model is slightly better at identifying students who will drop out while also reducing the number of students incorrectly flagged as at risk.

Although the neural network benefits slightly more from tuning than XGBoost, the magnitude of the improvement is still small compared with the large performance increase observed when moving from Stage 2 to Stage 3. This confirms that the primary driver of performance in Stage 3 is the addition of highly informative engineered features rather than changes to the model architecture.

Overall, once the Stage 3 variables are introduced, the neural network achieves very high accuracy with minimal sensitivity to hyperparameter choice, and tuning provides only incremental improvement.

## Stage 3 Model Comparison

### Performance Metrics Comparison (Stage 2 vs Stage 3)

Comparison of the results of the XGBoost and Neural network models on the Stage 3 dataset Vs the Stage 2 dataset

In [ ]:
# Stage 3 data model comparison

## list of tuned models
m_ids = [6, 8, 10, 12]

models_to_plot = build_models_to_plot(
    results,
    m_ids,
)
plot_roc_and_pr_curves(models_to_plot, figsize=(14, 7))





| Model          | Dataset | Accuracy   | Precision  | Recall     | Specificity | AUC        |
| -------------- | ------- | ---------- | ---------- | ---------- | ----------- | ---------- |
| XGBoost        | Stage 2 | 0.9168     | 0.7694     | 0.6352     | 0.9664      | 0.9340     |
| XGBoost        | Stage 3 | **0.9870** | **0.9751** | **0.9374** | **0.9958**  | **0.9989** |
| Neural Network | Stage 2 | 0.9094     | 0.7583     | 0.5806     | 0.9674      | 0.9220     |
| Neural Network | Stage 3 | **0.9862** | **0.9830** | **0.9241** | **0.9972**  | **0.9987** |


**Interpretation**

---

To ensure a fair comparison, the hyperparameters tuned on the Stage 2 dataset were reused as the baseline configuration when training both models on the Stage 3 dataset. This allows the effect of the additional Stage 3 features to be evaluated without the influence of further hyperparameter optimisation.

Both XGBoost and the neural network show a very large improvement when trained on the Stage 3 dataset. Accuracy, precision, recall, specificity, and AUC all increase substantially for both models, indicating that the additional features introduced in Stage 3 provide very strong predictive information.

In particular, both models become much better at identifying students who will drop out while still maintaining extremely high specificity, meaning that intervention efforts can be focused on genuinely at-risk students without incorrectly flagging large numbers of students who will complete.

XGBoost achieves the highest overall performance on the Stage 3 dataset, with slightly higher AUC, precision, and recall than the neural network. The ROC and Precision–Recall curves also show that the XGBoost model lies marginally outside the neural network across most thresholds. This suggests that the tree-based model is able to exploit the additional Stage 3 features slightly more effectively.

Overall, the results show that the major improvement in performance comes from the additional engineered features introduced in Stage 3 rather than from changes to the model architecture. Once these highly informative variables are included, both models perform extremely well, with XGBoost showing a small but consistent advantage in predictive accuracy.

---

### Effect of Stage 3 Hyperparameter Tuning

Hyperparameter tuning was repeated on the Stage 3 dataset for both the XGBoost and neural network models to assess whether further optimisation improves performance compared with the Stage 2 tuned configurations.

For XGBoost, tuning produces almost no change in performance. Accuracy, recall, precision, and AUC remain virtually identical, indicating that the Stage 2 hyperparameters were already close to optimal. Once the Stage 3 features are included, model performance becomes largely insensitive to further tuning.

The neural network shows a similar pattern. Retuning results in only small improvements, mainly in recall, while overall metrics and ROC / Precision–Recall curves remain very similar. This suggests that the Stage 2 configuration already provides a good fit when the stronger Stage 3 predictors are available.

Overall, hyperparameter tuning on the Stage 3 dataset provides minimal benefit for either model. The large performance increase observed in Stage 3 is driven by the additional engineered features rather than further optimisation of model hyperparameters.

# Stage 1, 2, 3 Tuned Model Comparison

In [ ]:
## list of tuned models
m_ids = [2, 4, 6, 8, 10, 12]

models_to_plot = build_models_to_plot(
    results,
    m_ids,
)
plot_roc_and_pr_curves(models_to_plot, figsize=(14, 7))


In [ ]:
# display all metrics from the model for final comparison in a table format 
# for easier review and comparison 

rows = []

for model_id, r in results.items():
    m = r["metrics"]

    rows.append(
        {
            "model_id": model_id,
            "stage": r["stage"],
            "model_type": r["model_type"],
            "model_name": r["model_name"],
            "accuracy": float(m["accuracy"]),
            "precision": float(m["precision"]),
            "recall": float(m["recall"]),
            "specificity": float(m["specificity"]),
            "auc": float(m["auc"]),
        }
    )

models_df = pd.DataFrame(rows)

models_df.sort_values(
    ["stage", "model_type", "model_name"],
    ascending=[True, False, True],
    inplace=True,
)

display(
    HTML(
        '<div style="max-height:360px; overflow:auto;">'
        + models_df.to_html(index=False)
        + "</div>"
    )
)


In [ ]:

# display the hyperparameters for all the models in a table to compare the 
# different configurations used for each model
def hyperparameter_summary(results: dict) -> pd.DataFrame:
    rows = []

    for m_id, r in results.items():

        hp = r.get("hyperparameters", {})

        row = {
            "stage": r.get("stage"),
            "model_type": r.get("model_type"),
            "model_name": r.get("model_name"),
        }

        # flatten hyperparameters
        for k, v in hp.items():
            row[k] = v

        rows.append(row)

    df = pd.DataFrame(rows)

    return df.sort_values(["stage", "model_type", "model_name"])

hp_df = hyperparameter_summary(results)

display(
    HTML(
        '<div style="max-height:360px; overflow:auto;">'
        + hp_df.to_html(index=False)
        + "</div>"
    )
)
